In [1]:
import numpy as np
import gmsh

In [2]:
mm = 1e-3

In [3]:
lc = 0.6
lc_fine = 0.25

gmsh.initialize()
gmsh.option.setNumber("General.Terminal", 0)

In [4]:
gmsh.model.add("mixing_elbow")
geo = gmsh.model.geo
s = lambda v: v * mm
cy = 200
R_out = 200.0
y_low = cy - np.sqrt(R_out**2 - 125.0**2)    # 43.875 -> x=325 sobre el arco
y_hi  = cy - np.sqrt(R_out**2 - 150.0**2)    # 67.712 -> x=350 sobre el ar
P = lambda x, y, h=lc: geo.addPoint(s(x), s(y), 0, s(h))
p1, p2 = P(0, 0), P(200, 0)
p3 = P(325, y_low, lc_fine)
p4, p5 = P(325, -25, lc_fine), P(350, -25, lc_fine)
p6 = P(350, y_hi, lc_fine)                   # antes (350, 75)
p7 = P(400, 200)                             # antes (400, 175)
p8,  p9  = P(400, 400),    P(300, 400)             # salida
p10, p11 = P(300, 200),    P(200, 100)             # arco superior
p12 = P(0, 100)
c   = geo.addPoint(s(200), s(cy), 0, s(lc))        # centro comun de los dos arcos

In [5]:
l_bottom = geo.addLine(p1, p2)                     # pared inferior del tubo horizontal
l_arc_lo = geo.addCircleArc(p2, c, p3)             # pared exterior del codo (R = 200)
l_side_l = geo.addLine(p3, p4)                     # pared izquierda entrada secundaria
l_in2    = geo.addLine(p4, p5)                     # ENTRADA secundaria (25 mm)
l_side_r = geo.addLine(p5, p6)
l_trans  = geo.addCircleArc(p6, c, p7)       # (350,67.7) -> (400,200), mismo centro c
l_right  = geo.addLine(p7, p8)               # tangente vertical, sin quiebre
l_out    = geo.addLine(p8, p9)                     # SALIDA (100 mm)
l_left   = geo.addLine(p9, p10)                    # pared izquierda del tubo vertical
l_arc_up = geo.addCircleArc(p10, c, p11)           # pared interior del codo (R = 100)
l_top    = geo.addLine(p11, p12)                   # pared superior del tubo horizontal
l_in1    = geo.addLine(p12, p1)                    # ENTRADA principal (100 mm)

walls = [l_bottom, l_arc_lo, l_side_l, l_side_r, l_trans, l_right, l_left, l_arc_up, l_top]

In [6]:
loop = geo.addCurveLoop(walls[:3] + [l_in2, l_side_r, l_trans, l_right, l_out, l_left, l_arc_up, l_top, l_in1])
surface = geo.addPlaneSurface([loop])
geo.synchronize()

In [7]:
gm = gmsh.model
gm.addPhysicalGroup(1, [l_in1],  name='inlet_main')
gm.addPhysicalGroup(1, [l_in2],  name='inlet_side')
gm.addPhysicalGroup(1, [l_out],  name='outlet')
gm.addPhysicalGroup(1, walls,    name='walls')
gm.addPhysicalGroup(2, [surface], name='fluid')

5

In [8]:
gmsh.option.setNumber("Mesh.Algorithm", 6)         # Frontal-Delaunay: triangulos regulares
gm.mesh.generate(2)
gm.mesh.setOrder(2)                                # T6 -> el proyecto usa Taylor-Hood P2/P1

In [9]:

import sys


if 'close' not in sys.argv:
    gmsh.fltk.run()